# TrustLens — Day 1 EDA & Tier-1 Heuristic Baseline

Short exploratory pass over the data and the Tier-1 heuristic features
(ELA, EXIF, FFT). Goal: confirm the features carry signal and establish the
**recall @ fixed FPR** baseline every later model must beat.

> Datasets (140k faces, MIDV-500) are auth-gated / multi-GB. This notebook runs
> on **synthetic fixtures** so it works offline. Point `DATA` at
> `data/raw/140k_faces/...` once downloaded to rerun on real images.


In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from trustlens.data.fixtures import generate_fixtures
from trustlens.heuristics.classifier import (
    HeuristicClassifier, extract_features, features_to_vector, FEATURE_NAMES)
from trustlens.metrics import evaluate_at_fpr, roc_points, pr_points, roc_auc
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

## 1. Load data (synthetic fixtures)

In [ ]:
manifest = generate_fixtures("../data/interim/fixtures", n_per_class=60, size=256, seed=0)
paths = [p for p, _ in manifest]
y = np.array([l for _, l in manifest])
print(f"{len(paths)} images | class balance: real={np.sum(y==0)} fake={np.sum(y==1)}")

Peek at one image per class.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
ax[0].imshow(Image.open(paths[0]));  ax[0].set_title("real (label 0)"); ax[0].axis("off")
ax[1].imshow(Image.open(paths[1]));  ax[1].set_title("fake (label 1)"); ax[1].axis("off")
plt.tight_layout()

## 2. Extract Tier-1 features

In [ ]:
rows = [extract_features(Image.open(p)) for p in paths]
df = pd.DataFrame(rows); df["label"] = y
X = df[FEATURE_NAMES].to_numpy()
df.groupby("label")[FEATURE_NAMES].mean().T

## 3. Feature separability

Class-conditional distributions. EXIF presence and the FFT peakiness/high-freq
signals should separate the classes most cleanly.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, feat in zip(axes.ravel(), FEATURE_NAMES):
    for lab, name in [(0, "real"), (1, "fake")]:
        ax.hist(df.loc[df.label==lab, feat], bins=20, alpha=0.6, label=name)
    ax.set_title(feat); ax.legend()
plt.tight_layout()

## 4. Baseline: rule-based default vs calibrated

In [ ]:
default = HeuristicClassifier.default()
scores_d = np.array([default.predict_proba_features(features_to_vector(r))[0] for r in rows])

calibrated = HeuristicClassifier().fit(X, y)
scores_c = calibrated.predict_proba_features(X)

for name, s in [("rule-based default", scores_d), ("calibrated", scores_c)]:
    r = evaluate_at_fpr(y, s, target_fpr=0.01)
    print(f"{name:20s}  AUC={roc_auc(y, s):.3f}  recall@1%FPR={r.recall:.3f}  "
          f"precision={r.precision:.3f}  thr={r.threshold:.3f}")

## 5. ROC & PR curves (the operating-point picture)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, s in [("default", scores_d), ("calibrated", scores_c)]:
    fpr, tpr = roc_points(y, s); ax[0].plot(fpr, tpr, marker=".", label=name)
    rec, prec = pr_points(y, s); ax[1].plot(rec, prec, marker=".", label=name)
ax[0].axvline(0.01, ls="--", c="grey", lw=1, label="target FPR=1%")
ax[0].set(xlabel="FPR", ylabel="TPR (recall)", title="ROC"); ax[0].legend()
ax[1].set(xlabel="recall", ylabel="precision", title="PR"); ax[1].legend()
plt.tight_layout()

## 6. Takeaways

- All three heuristic families contribute; EXIF presence + FFT peakiness are the
  strongest separators on these fixtures.
- The **rule-based default already saturates** the synthetic benchmark — exactly
  the "ship the simple thing first" result we wanted for Day 1.
- On real data expect lower numbers; the **recall @ 1% FPR** figure is the
  baseline Day-2 learned models are graded against via `metrics.evaluate_at_fpr`.
- Next: rerun this notebook on the real 140k-faces test split and MIDV-500 frames.
